In [14]:
import pandas as pd

df = pd.read_csv("../Data/Modified Dataset/Dataset_with_Weather_features.csv")
print("Shape:", df.shape)
df.head()

Shape: (51862, 14)


,Date,Start_Hour,End_Hour,Source,Day_of_Year,Day_Name,Month_Name,Season,Production,Temperature_C,Humidity_Percent,Precipitation_mm,WindSpeed_kmh,Rainfall_Flag
0,2025-11-30,21,22,Wind,334,Sunday,November,Fall,5281,11.8,72,0.0,5.9,No
1,2025-11-30,18,19,Wind,334,Sunday,November,Fall,3824,13.5,67,0.0,7.2,No
2,2025-11-30,16,17,Wind,334,Sunday,November,Fall,3824,17.3,50,0.0,5.1,No
3,2025-11-30,23,0,Wind,334,Sunday,November,Fall,6120,10.4,69,0.0,5.4,No
4,2025-11-30,6,7,Wind,334,Sunday,November,Fall,4387,8.2,62,0.0,1.6,No


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51862 entries, 0 to 51861
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              51862 non-null  str    
 1   Start_Hour        51862 non-null  int64  
 2   End_Hour          51862 non-null  int64  
 3   Source            51862 non-null  str    
 4   Day_of_Year       51862 non-null  int64  
 5   Day_Name          51862 non-null  str    
 6   Month_Name        51862 non-null  str    
 7   Season            51862 non-null  str    
 8   Production        51862 non-null  int64  
 9   Temperature_C     51862 non-null  float64
 10  Humidity_Percent  51862 non-null  int64  
 11  Precipitation_mm  51862 non-null  float64
 12  WindSpeed_kmh     51862 non-null  float64
 13  Rainfall_Flag     51862 non-null  str    
dtypes: float64(3), int64(5), str(6)
memory usage: 5.5 MB


In [16]:
df.describe(include='all')

,Date,Start_Hour,End_Hour,Source,Day_of_Year,Day_Name,Month_Name,Season,Production,Temperature_C,Humidity_Percent,Precipitation_mm,WindSpeed_kmh,Rainfall_Flag
count,51862,51862.000000,51862.000000,51862,51862.000000,51862,51862,51862,51862.000000,51862.000000,51862.000000,51862.000000,51862.000000,51862
unique,2161,NaN,NaN,2,NaN,7,12,4,NaN,NaN,NaN,NaN,NaN,2
top,2025-10-26,NaN,NaN,Wind,NaN,Sunday,October,Summer,NaN,NaN,NaN,NaN,NaN,No
freq,25,NaN,NaN,42484,NaN,7416,4470,13247,NaN,NaN,NaN,NaN,NaN,46073
mean,NaN,11.499711,11.499672,NaN,180.800278,NaN,NaN,NaN,6215.242625,21.592575,59.241256,0.124289,7.248243,NaN
std,NaN,6.922230,6.922186,NaN,104.292759,NaN,NaN,NaN,3978.339604,8.110513,19.983908,0.721164,3.856219,NaN
min,NaN,0.000000,0.000000,NaN,1.000000,NaN,NaN,NaN,58.000000,1.800000,7.000000,0.000000,0.000000,NaN
25%,NaN,5.250000,5.250000,NaN,91.000000,NaN,NaN,NaN,3111.000000,15.000000,45.000000,0.000000,4.700000,NaN
50%,NaN,11.000000,11.000000,NaN,181.000000,NaN,NaN,NaN,5372.000000,22.600000,60.000000,0.000000,6.600000,NaN
75%,NaN,17.000000,17.000000,NaN,271.000000,NaN,NaN,NaN,8501.000000,27.700000,75.000000,0.000000,9.100000,NaN


In [17]:
print("Missing values per column:")
df.isnull().sum()

Missing values per column:


Date                0
Start_Hour          0
End_Hour            0
Source              0
Day_of_Year         0
Day_Name            0
Month_Name          0
Season              0
Production          0
Temperature_C       0
Humidity_Percent    0
Precipitation_mm    0
WindSpeed_kmh       0
Rainfall_Flag       0
dtype: int64

In [18]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [19]:
df['Source'].value_counts()

Source
Wind     42484
Solar     9378
Name: count, dtype: int64

In [20]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_cols = ["Start_Hour", "End_Hour", "Day_of_Year", "Production"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Rows with conversion errors (NaN after type fix):")
df[numeric_cols + ["Date"]].isnull().sum()

Rows with conversion errors (NaN after type fix):


Start_Hour     0
End_Hour       0
Day_of_Year    0
Production     0
Date           0
dtype: int64

In [21]:
before = len(df)

# Dropping rows with Source = 'Mixed' 
df = df[df['Source'] != 'Mixed']

# Dropping exact duplicate rows 
df = df.drop_duplicates()

# Dropping rows with missing critical values
df = df.dropna(subset=["Date", "Production", "Source"])

# Removing invalid hour values (should be 0-23)
df = df[(df["Start_Hour"].between(0, 23)) & (df["End_Hour"].between(0, 23))]

# Removing invalid/negative production values
df = df[df["Production"] >= 0]

# Removing invalid Day_of_Year values (should be 1-366)
df = df[df["Day_of_Year"].between(1, 366)]

# Humidity should be a valid percentage (0-100)
df = df[df["Humidity_Percent"].between(0, 100)]

# Precipitation cannot be negative
df = df[df["Precipitation_mm"] >= 0]

# Wind speed cannot be negative
df = df[df["WindSpeed_kmh"] >= 0]

# Rainfall_Flag should only be "Yes" or "No"
df = df[df["Rainfall_Flag"].isin(["Yes", "No"])]

after = len(df)
print(f"Rows removed during cleaning: {before - after}")
print(f"Remaining rows: {after}")

Rows removed during cleaning: 0
Remaining rows: 51862


In [22]:
categorical_cols = ["Source", "Season", "Day_Name", "Month_Name", "Rainfall_Flag"]

for col in categorical_cols:
    print(f"\nUnique values in '{col}':")
    print(df[col].value_counts())


Unique values in 'Source':
Source
Wind     42484
Solar     9378
Name: count, dtype: int64

Unique values in 'Season':
Season
Summer    13247
Spring    13241
Fall      13110
Winter    12264
Name: count, dtype: int64

Unique values in 'Day_Name':
Day_Name
Sunday       7416
Saturday     7416
Friday       7416
Thursday     7416
Wednesday    7414
Tuesday      7392
Monday       7392
Name: count, dtype: int64

Unique values in 'Month_Name':
Month_Name
October      4470
August       4464
July         4464
May          4464
January      4464
March        4458
November     4320
September    4320
June         4319
April        4319
February     4080
December     3720
Name: count, dtype: int64

Unique values in 'Rainfall_Flag':
Rainfall_Flag
No     46073
Yes     5789
Name: count, dtype: int64


In [23]:
# Keeping a readable copy for EDA 
df_for_eda = df.copy()

# One-hot encoding for modeling 
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Converting boolean columns to integers (0/1) for cleaner model compatibility
bool_cols = df_encoded.select_dtypes('bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

df_encoded.head()

,Date,Start_Hour,End_Hour,Day_of_Year,Production,Temperature_C,Humidity_Percent,Precipitation_mm,WindSpeed_kmh,Source_Wind,...,Month_Name_February,Month_Name_January,Month_Name_July,Month_Name_June,Month_Name_March,Month_Name_May,Month_Name_November,Month_Name_October,Month_Name_September,Rainfall_Flag_Yes
0,2025-11-30,21,22,334,5281,11.8,72,0.0,5.9,1,...,0,0,0,0,0,0,1,0,0,0
1,2025-11-30,18,19,334,3824,13.5,67,0.0,7.2,1,...,0,0,0,0,0,0,1,0,0,0
2,2025-11-30,16,17,334,3824,17.3,50,0.0,5.1,1,...,0,0,0,0,0,0,1,0,0,0
3,2025-11-30,23,0,334,6120,10.4,69,0.0,5.4,1,...,0,0,0,0,0,0,1,0,0,0
4,2025-11-30,6,7,334,4387,8.2,62,0.0,1.6,1,...,0,0,0,0,0,0,1,0,0,0


In [26]:
df_for_eda.to_csv("../Data/CLeaned/Cleaned_Readable_Data.csv", index=False)
df_encoded.to_csv("../Data/Cleaned/Cleaned_Production_Data.csv", index=False)

print("Saved: Data/Cleaned/Cleaned_Readable_Data.csv (for EDA)")
print("Saved: Data/Cleaned/Cleaned_Production_Data.csv (encoded, for modeling)")
print("\nFinal encoded columns:")
print(df_encoded.columns.tolist())

Saved: Data/Cleaned/Cleaned_Readable_Data.csv (for EDA)
Saved: Data/Cleaned/Cleaned_Production_Data.csv (encoded, for modeling)

Final encoded columns:
['Date', 'Start_Hour', 'End_Hour', 'Day_of_Year', 'Production', 'Temperature_C', 'Humidity_Percent', 'Precipitation_mm', 'WindSpeed_kmh', 'Source_Wind', 'Season_Spring', 'Season_Summer', 'Season_Winter', 'Day_Name_Monday', 'Day_Name_Saturday', 'Day_Name_Sunday', 'Day_Name_Thursday', 'Day_Name_Tuesday', 'Day_Name_Wednesday', 'Month_Name_August', 'Month_Name_December', 'Month_Name_February', 'Month_Name_January', 'Month_Name_July', 'Month_Name_June', 'Month_Name_March', 'Month_Name_May', 'Month_Name_November', 'Month_Name_October', 'Month_Name_September', 'Rainfall_Flag_Yes']
